# NFL Prediction Model V2 — Local Workbench

This notebook uses the **existing V2 scripts and models**, not a new prediction implementation. Start it by double-clicking `start_jupyter.command` on macOS. Run the cells one by one.

**Warning:** The V2 rolling-feature updater has not been confirmed against the original model-training feature engineering. Review outputs before relying on predictions.


In [1]:
from pathlib import Path
import sys, subprocess, datetime
ROOT = Path.cwd()
print('Project:', ROOT)
print('Python:', sys.executable)
assert (ROOT / 'ML_predict_2.py').exists(), 'Run the notebook from the project folder'


Project: /Users/aaronbradin/Desktop/NFL_Prediction_Model_V2 4
Python: /Users/aaronbradin/Desktop/NFL_Prediction_Model_V2 4/.venv/bin/python


## 1. Check model files
Verifies the original LightGBM models and 394-feature mapping are present and dimensionally compatible.


In [2]:
subprocess.run([sys.executable, 'check_artifacts.py'], cwd=ROOT, check=True)


All required artifacts present and feature dimensions match.


CompletedProcess(args=['/Users/aaronbradin/Desktop/NFL_Prediction_Model_V2 4/.venv/bin/python', 'check_artifacts.py'], returncode=0)

## 2. Refresh completed-game data (optional)
This runs the existing weekly updater. **It changes `artifacts/team_feature_store.parquet`** if new valid team-weeks exist. It does **not** retrain either model. A data-source schema mismatch will raise an error rather than silently create fabricated features.


In [3]:
season = datetime.datetime.now().year
print('Refreshing season', season)
subprocess.run([sys.executable, 'weekly_refresh.py', '--season', str(season)], cwd=ROOT, check=True)


Refreshing season 2026


/Users/aaronbradin/Desktop/NFL_Prediction_Model_V2 4/weekly_update.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  updated[col]=source.groupby('team_full')[base].transform(lambda s:s.rolling(int(window),min_periods=1).mean())
/Users/aaronbradin/Desktop/NFL_Prediction_Model_V2 4/weekly_update.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  updated[col]=source.groupby('team_full')[base].transform(lambda s:s.rolling(int(window),min_periods=1).mean())
/Users/aaronbradin/Desktop/NFL_Prediction_Model_V2 4/weekly_update.py:

Info: preserving legitimately missing kicking stats: fg_long, fg_pct, pat_pct, pt_long
{'season': 2026, 'appended_rows': 124, 'through_week': 7, 'status': 'updated'}
          home_full            away_full       favorite_team  favorite_win_prob  features_asof_home_week  features_asof_away_week
      Chicago Bears New England Patriots       Chicago Bears           0.884217                        6                        6
 New Orleans Saints  Pittsburgh Steelers  New Orleans Saints           0.535427                        6                        6
    Atlanta Falcons  San Francisco 49ers     Atlanta Falcons           0.567447                        6                        6
   Baltimore Ravens   Cincinnati Bengals    Baltimore Ravens           0.581302                        6                        5
  Carolina Panthers Tampa Bay Buccaneers   Carolina Panthers           0.554530                        6                        6
     Houston Texans      New York Giants      Houston 

CompletedProcess(args=['/Users/aaronbradin/Desktop/NFL_Prediction_Model_V2 4/.venv/bin/python', 'weekly_refresh.py', '--season', '2026'], returncode=0)

## 3. Launch your unchanged Streamlit dashboard
Run the cell below. Then click http://localhost:8501 and use **Refresh predictions** in the original dashboard. You can keep Jupyter open in a separate browser tab.


In [4]:
import subprocess, time
try:
    streamlit_process
    already_running = streamlit_process.poll() is None
except NameError:
    already_running = False
if not already_running:
    streamlit_process = subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'ML_predict_2.py', '--server.address', '127.0.0.1', '--server.port', '8501', '--server.headless', 'true'], cwd=ROOT)
    time.sleep(2)
print('Dashboard: http://localhost:8501')
print('Streamlit process ID:', streamlit_process.pid)


2026-10-23 09:06:22.281 Uvicorn server started on 127.0.0.1:8501



  You can now view your Streamlit app in your browser.

  URL: http://127.0.0.1:8501

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
Dashboard: http://localhost:8501
Streamlit process ID: 72708


## 4. Stop Streamlit when finished (optional)


In [ ]:
if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    print('Stopped Streamlit')
else:
    print('Streamlit is not running from this notebook')


## 5. Publish updated V2 project to GitHub

**First run:** Create an empty GitHub repository and set its URL below, then run the cell. On later runs, leave `repo_url` blank. This stages only approved project files, commits changes, and pushes to the configured remote. If the remote already has its own commits, use a clean clone or merge first; the script will not force-push. Keep Streamlit Cloud pointed at `ML_predict_2.py` in the chosen repo/branch.


In [5]:
repo_url = ""  # First run: "https://github.com/YOUR_USERNAME/YOUR_NEW_V2_REPO.git"
command = [sys.executable, 'publish_github.py']
if repo_url.strip():
    command += ['--repo-url', repo_url.strip()]
subprocess.run(command, cwd=ROOT, check=True)


Not a git checkout. First create an EMPTY GitHub repository, then run: python publish_github.py --repo-url https://github.com/USERNAME/REPO.git


CalledProcessError: Command '['/Users/aaronbradin/Desktop/NFL_Prediction_Model_V2 4/.venv/bin/python', 'publish_github.py']' returned non-zero exit status 1.